# pyologger workshop demo (NDP)

A self-contained walkthrough of the `pyologger` tag-data pipeline, built to run inside a fresh **National Data Platform (NDP)** workspace with no prior setup, no external drives, and no lab database access.

**What you'll do:**
1. Download the workshop's demo dataset (a Google Drive zip) and run `setup_demo.py` to install it — the same setup used by the main workshop guide (`demo_guide.md`).
2. Load it as a `DataReader` object, the same class the full pipeline uses.
3. Explore what's inside: signals, channels, sampling rates, and scored sleep-state events.
4. Build an interactive multi-signal dashboard — depth, EEG (with a live spectrogram), ECG, heart rate, stroke rate — with sleep states shaded on top.
5. Zoom into a single REM bout at full resolution.

**Example deployment:** `2020-04-10_mian-002` — a juvenile northern elephant seal (*Mirounga angustirostris*, nicknamed "SnoozySuzy") carrying EEG/EOG/EMG, ECG, and a motion + depth tag, recorded on land at Año Nuevo State Park. The full recording is 4.2 days; this notebook loads a pre-trimmed 3-hour demo slice (2020-04-12 09:00–12:00 local) so downloads and rendering stay fast.

> This notebook intentionally skips the metadata (Notion) and raw-import steps used internally by the lab — those need private credentials. Everything here comes from one self-contained public download.

**Just run all cells top to bottom** (Run All, or Shift+Enter through each one) — every setup step below checks whether it's already done and skips itself if so, so re-running the whole notebook (or just the cells you need) is always safe.

## 0. One-time setup

Installs `pyologger` and its notebook dependencies into whatever Python environment this notebook's kernel is running (no separate venv or terminal needed). Safe to re-run — pip skips anything already installed at the right version.

In [ ]:
%pip install -e .. -q
%pip install plotly plotly-resampler pandas xarray gdown -q

## 1. Download and set up the demo dataset

The cell below downloads the workshop's demo dataset zip
([Google Drive link](https://drive.google.com/file/d/1zqO8pzu48BWOJp2OlvllIueT_P0jloCi/view?usp=sharing),
~420 MB) straight into a folder that sits *next to* this repo (not inside it), unzips it, then
runs `setup_demo.py` — the same script and `demo_config.yaml` the main workshop guide
(`demo_guide.md`) uses, so there's one source of truth for how the demo gets set up:

1. Downloads `NSF_demo_data.zip` via `gdown` (handles Google Drive's large-file
   confirmation step automatically)
2. Unzips it into a sibling folder next to this repo
3. Runs `setup_demo.py`, which finds those files, builds the dataset/deployment
   hierarchy pyologger expects in a new sibling folder `pyologger_demo_data/`,
   installs `demo_config.yaml` as `config.yaml` (only if `config.yaml` doesn't
   already exist), and installs `.env` from `.env.example` (only if `.env`
   doesn't already exist)

No Pelican token, Notion token, or lab credentials are needed for any step in this notebook.

In [ ]:
import os
import zipfile

import gdown

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
download_dir = os.path.abspath(os.path.join(repo_root, "..", "NSF_demo_data_download"))
extracted_dir = os.path.join(download_dir, "NSF_demo_data")
os.makedirs(download_dir, exist_ok=True)

GDRIVE_FILE_ID = "1zqO8pzu48BWOJp2OlvllIueT_P0jloCi"
zip_path = os.path.join(download_dir, "NSF_demo_data.zip")

# Safe to re-run: skip the download/unzip if the files are already there.
if os.path.isdir(extracted_dir) and os.listdir(extracted_dir):
    print(f"Demo files already extracted at {extracted_dir} -- skipping download.")
else:
    if os.path.exists(zip_path):
        print(f"Zip already downloaded at {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")
    else:
        gdown.download(id=GDRIVE_FILE_ID, output=zip_path, quiet=False)
        print(f"Downloaded to {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")

    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(download_dir)
    print(f"Unzipped into {download_dir}")

In [ ]:
import os
import subprocess
import sys

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
dataset_id = "mian-juv-nese_sleep_lml-ano_JKB"
deployment_id = "2020-04-10_mian-002"

demo_data_dir = os.path.abspath(os.path.join(repo_root, "..", "pyologger_demo_data"))
deployment_outputs_dir = os.path.join(demo_data_dir, dataset_id, deployment_id, "outputs")
pkl_local_path = os.path.join(deployment_outputs_dir, "data.pkl")

# setup_demo.py itself is safe to re-run (it skips config.yaml/.env if they
# already exist, and skips any deployment file it can't find in the download
# folder -- which is expected once a previous run has already moved them into
# pyologger_demo_data/). This check just gives a clear message either way and
# avoids re-running the subprocess at all once everything is already in place.
if os.path.exists(pkl_local_path):
    print(f"Demo data already set up at {deployment_outputs_dir} -- skipping setup_demo.py.")
else:
    result = subprocess.run(
        [sys.executable, os.path.join(repo_root, "setup_demo.py")],
        cwd=repo_root, capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(
            "setup_demo.py failed -- check the output above. Make sure the "
            "download+unzip cell above completed successfully first."
        )
    if not os.path.exists(pkl_local_path):
        raise RuntimeError(
            f"setup_demo.py ran but {pkl_local_path} still doesn't exist -- "
            "check the output above for a [warn]/[skip] line explaining why."
        )

config_path = os.path.join(repo_root, "config.yaml")
print(f"config.yaml       : {config_path}")
print(f"Demo data folder  : {demo_data_dir}")
print(f"data.pkl          : {pkl_local_path}")

## 2. Load the deployment

This `.pkl` is a pickled `pyologger.load_data.datareader.DataReader` object — already imported, calibrated, and derived (zero-offset-corrected depth, oriented accelerometer/magnetometer, detected heartbeats/strokes, and scored sleep states). It's the exact same object type produced by the pipeline's `read_files()` step, just further downstream, so every plotting/analysis utility in `pyologger` works on it unchanged.

In [ ]:
sys.path.insert(0, repo_root)

import pickle
import pandas as pd

from pyologger.utils.folder_manager import load_configuration
from pyologger.utils.param_manager import ParamManager
from pyologger.plot_data.plotter import plot_tag_data_interactive, _expand_state_annotation_patterns, load_color_mapping

config, data_dir, color_mapping_path, montage_path = load_configuration()

with open(pkl_local_path, "rb") as f:
    data_pkl = pickle.load(f)

# The pkl was written with a stale output_folder from the original machine; repoint
# it at where we actually loaded it from.
data_pkl.output_folder = deployment_outputs_dir
data_pkl.deployment_folder = os.path.dirname(deployment_outputs_dir)

param_manager = ParamManager(deployment_folder=data_pkl.deployment_folder, deployment_id=deployment_id)

timezone = data_pkl.deployment_info["Time Zone"]
print(f"Deployment : {deployment_id}")
print(f"Animal     : {dataset_id}")
print(f"Time zone  : {timezone}")

## 3. What's inside?

Every signal lives in `data_pkl.signal_data` as a `pandas.DataFrame` with a `datetime` column, and its metadata (units, sampling rate, channel names) in `data_pkl.signal_info`.

In [ ]:
rows = []
for name, df in data_pkl.signal_data.items():
    if df is None or "datetime" not in df.columns or df.empty:
        continue
    rows.append({
        "signal": name,
        "samples": len(df),
        "channels": ", ".join(c for c in df.columns if c != "datetime"),
        "start": df["datetime"].iloc[0],
        "end": df["datetime"].iloc[-1],
    })

signal_overview = pd.DataFrame(rows).sort_values("samples", ascending=False).reset_index(drop=True)
signal_overview

Sleep states are stored in `data_pkl.event_data` as rows with `type == "state"`, each with a `datetime` (onset) and a `duration` in seconds. Two parallel scoring schemes are present: a detailed scheme (active/quiet waking, LV-SWS, HV-SWS, certain/putative REM) and a collapsed "simple" scheme (active waking, quiet waking, SWS, REM).

In [ ]:
events = data_pkl.event_data
state_events = events[events["type"] == "state"].copy()

summary = (
    state_events.groupby("key")
    .agg(n_events=("key", "size"), total_seconds=("duration", "sum"))
    .sort_values("total_seconds", ascending=False)
)
summary["total_minutes"] = (summary["total_seconds"] / 60).round(1)
summary[["n_events", "total_minutes"]]

## 4. Interactive dashboard

`plot_tag_data_interactive` returns a `plotly-resampler` figure: it renders a downsampled view and re-fetches full-resolution data as you zoom — what makes a multi-million-sample ECG trace usable in a notebook.

State annotations shade time spans (sleep states); note annotations mark individual points (detected heartbeats, strokes, dives). Colors and target signal rows both resolve automatically from `color_mappings.json`, which ships in this repo.

Since this NDP workspace is a JupyterHub-spawned pod (not your own machine), the dashboard is shown through the Hub's proxy rather than a plain `localhost` port — see `show_inline_dashboard` below.

In [ ]:
# shade_mode="all_y" is set explicitly here because color_mappings.json has no
# __event_styles__ entry yet for these sleep-state keys -- without it, the plotter
# falls back to drawing a thin line at the signal's median instead of a shaded band.
state_annotations_detailed = {"sleep-state_*": {"shade_mode": "all_y"}}       # -> shaded on the EEG row
state_annotations_simple = {"simple_sleep_state.*": {"shade_mode": "all_y"}}  # -> shaded on the depth row
state_annotations = {**state_annotations_detailed, **state_annotations_simple}

notes_to_plot = {
    "heartbeat_auto_detect_accepted":  {"signal": "heart_rate",  "symbol": "triangle-up",   "color": "green"},
    "strokebeat_auto_detect_accepted": {"signal": "stroke_rate", "symbol": "triangle-up",   "color": "green"},
    "dive":                            {"signal": "depth",       "symbol": "triangle-down", "color": "blue"},
}

color_mapping = load_color_mapping(color_mapping_path)
available_keys = sorted(data_pkl.event_data["key"].astype(str).unique())
for key in _expand_state_annotation_patterns(state_annotations, available_keys):
    print(f"{key:42s} {color_mapping.get(key)}")

In [ ]:
from urllib.parse import urlparse

def show_inline_dashboard(fig, port=8050, height=900):
    """Show a plotly-resampler figure as an inline Dash app.

    This NDP workspace is a JupyterHub-spawned pod, so ports aren't reachable
    via localhost from the browser. The Hub's singleuser server runs
    jupyter-server-proxy, which exposes any local port at
    <hub>/user/<you>/proxy/<port>/ and transparently strips that prefix before
    forwarding to the local server -- Dash only needs to know the prefix for
    the links/assets it writes into the page, not for its own routing.
    (see https://github.com/predict-idlab/plotly-resampler/issues/265)
    """
    hub_prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX")
    proxy_uri = os.environ.get("VSCODE_PROXY_URI")  # only used to recover the hub's public scheme/host
    if hub_prefix and proxy_uri:
        hub_origin = urlparse(proxy_uri.replace("{{port}}", str(port)))
        server_url = f"{hub_origin.scheme}://{hub_origin.netloc}"
        requests_pathname_prefix = f"{hub_prefix}proxy/{port}/"
        print(f"Dashboard URL (open directly in a new tab to test): {server_url}{requests_pathname_prefix}")
        fig.show_dash(
            mode="inline",
            port=port,
            config={},  # dash-core-components' Graph crashes if config is left as None
            jupyter_height=height,
            jupyter_server_url=server_url,
            init_dash_kwargs={"requests_pathname_prefix": requests_pathname_prefix},
        )
    else:
        fig.show_dash(mode="inline", port=port, config={}, jupyter_height=height)

TARGET_SAMPLING_RATE = 10  # Hz for the initial (zoomed-out) render

fig = plot_tag_data_interactive(
    data_pkl=data_pkl,
    signals=["depth", "eeg", "ecg", "heart_rate", "stroke_rate", "odba", "prh"],
    channels={
        "eeg": ["eeg_p3", "eeg_p4", "eeg_f3", "eeg_f4"],
        "prh": ["pitch", "roll"],
    },
    state_annotations=state_annotations,
    note_annotations=notes_to_plot,
    color_mapping_path=color_mapping_path,
    target_sampling_rate=TARGET_SAMPLING_RATE,
    zoom_range_selector_channel="depth",
    spectrogram_channel="eeg_p4",   # STFT computed from this channel, drawn above its row
    spectrogram_range=(0, 10),      # delta + theta band — sleep EEG is mostly under 10 Hz
    spectrogram_contrast=(2, 98),   # dB percentiles used as color limits
)

show_inline_dashboard(fig, port=8050)

## 5. Zoom into a single REM bout

Pass `zoom_start_time` / `zoom_end_time` to open the figure already zoomed into a window, while keeping the surrounding context available when you zoom back out.

In [ ]:
rem_events = state_events[state_events["key"].str.contains("rem", case=False, na=False)]

if not rem_events.empty:
    bout = rem_events.loc[rem_events["duration"].idxmax()]
    bout_start = bout["datetime"]
    bout_end = bout_start + pd.Timedelta(seconds=float(bout["duration"]))
    print(f"Longest REM bout: {bout['key']}")
    print(f"  {bout_start} -> {bout_end}  ({bout['duration']:.0f} s)")

    pad = pd.Timedelta(minutes=5)
    ZOOM_START, ZOOM_END = bout_start - pad, bout_end + pad
else:
    print("No REM events in this window; falling back to the first 30 minutes.")
    ZOOM_START = data_pkl.signal_data["depth"]["datetime"].iloc[0]
    ZOOM_END = ZOOM_START + pd.Timedelta(minutes=30)

In [ ]:
fig_bout = plot_tag_data_interactive(
    data_pkl=data_pkl,
    signals=["depth", "eeg", "eog", "ecg", "heart_rate"],
    channels={
        "eeg": ["eeg_p3", "eeg_p4", "eeg_f3", "eeg_f4"],
        "eog": ["eog_l", "eog_r"],
    },
    state_annotations=state_annotations_detailed,
    color_mapping_path=color_mapping_path,
    target_sampling_rate=100,   # higher rate for a short span
    zoom_start_time=ZOOM_START,
    zoom_end_time=ZOOM_END,
    zoom_range_selector_channel="depth",
    state_annotation_channel_mode="combined",
    state_annotation_channel_height_ratio=0.15,
)

show_inline_dashboard(fig_bout, port=8051)

## Where to go from here

- Drag your own tag data into this workspace's file browser and point `dataset_folder` / `deployment_id` at it, then walk through `00_load_data.ipynb` → `01_calibrate_pressure.ipynb` → `02_calibrate_accmag.ipynb` for the full raw-to-processed pipeline this demo skipped.
- Everything you downloaded lives under `pyologger_demo_data/` next to this repo — safe to delete and re-run `setup_demo.py` from scratch at any time.
- Try the optional Step 7 in `demo_guide.md` (feature generation, clustering, and a sleep-stage random forest with channel ablation) to see how much EEG/heart-rate data actually helps classify sleep stage.
- The lab's Pelican namespaces (`jkb-lab`, `jkb-lab-public`) work well for publishing/pulling your own data once you have a token — see the `/pelican` skill or the Pelican Wiki. (Not used by this demo — see `demo_guide.md`'s Known Issues for why.)